# Chapter 11 — Training Deep Neural Networks
Companion notebook to `part2_deep_learning/11_training_deep_neural_nets.md`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras

fashion_mnist = keras.datasets.fashion_mnist
(X_train_full, y_train_full), (X_test, y_test) = fashion_mnist.load_data()
X_valid, X_train = X_train_full[:5000] / 255.0, X_train_full[5000:] / 255.0
y_valid, y_train = y_train_full[:5000], y_train_full[5000:]
X_test = X_test / 255.0

## He init + ELU + Batch Normalization + Nadam

In [ ]:
def build_model():
    return keras.models.Sequential([
        keras.layers.Flatten(input_shape=[28, 28]),
        keras.layers.BatchNormalization(),
        keras.layers.Dense(300, activation="elu", kernel_initializer="he_normal"),
        keras.layers.BatchNormalization(),
        keras.layers.Dense(100, activation="elu", kernel_initializer="he_normal"),
        keras.layers.BatchNormalization(),
        keras.layers.Dense(10, activation="softmax"),
    ])

model_bn = build_model()
model_bn.compile(loss="sparse_categorical_crossentropy", optimizer="nadam", metrics=["accuracy"])
history_bn = model_bn.fit(X_train, y_train, epochs=10, validation_data=(X_valid, y_valid))

## Dropout as regularization

In [ ]:
model_dropout = keras.models.Sequential([
    keras.layers.Flatten(input_shape=[28, 28]),
    keras.layers.Dropout(rate=0.2),
    keras.layers.Dense(300, activation="elu", kernel_initializer="he_normal"),
    keras.layers.Dropout(rate=0.2),
    keras.layers.Dense(100, activation="elu", kernel_initializer="he_normal"),
    keras.layers.Dropout(rate=0.2),
    keras.layers.Dense(10, activation="softmax"),
])
model_dropout.compile(loss="sparse_categorical_crossentropy", optimizer="nadam", metrics=["accuracy"])
history_dropout = model_dropout.fit(X_train, y_train, epochs=10, validation_data=(X_valid, y_valid))

## Callbacks: ModelCheckpoint + EarlyStopping, and a learning-rate schedule

In [ ]:
model_cb = build_model()
model_cb.compile(loss="sparse_categorical_crossentropy", optimizer="nadam", metrics=["accuracy"])

checkpoint_cb = keras.callbacks.ModelCheckpoint("best_model.h5", save_best_only=True)
early_stopping_cb = keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)

history_cb = model_cb.fit(X_train, y_train, epochs=50,
                           validation_data=(X_valid, y_valid),
                           callbacks=[checkpoint_cb, early_stopping_cb])
print("Stopped after", len(history_cb.history["loss"]), "epochs")

In [ ]:
def exponential_decay(lr0, s):
    def fn(epoch):
        return lr0 * 0.1 ** (epoch / s)
    return fn

lr_scheduler = keras.callbacks.LearningRateScheduler(exponential_decay(lr0=0.01, s=20))
model_lr = build_model()
model_lr.compile(loss="sparse_categorical_crossentropy",
                  optimizer=keras.optimizers.SGD(learning_rate=0.01), metrics=["accuracy"])
history_lr = model_lr.fit(X_train, y_train, epochs=15,
                           validation_data=(X_valid, y_valid), callbacks=[lr_scheduler])

## Compare all variants

In [ ]:
for name, h in [("BatchNorm+Nadam", history_bn), ("Dropout", history_dropout),
                ("Checkpoint+EarlyStop", history_cb), ("LR schedule", history_lr)]:
    print(f"{name:22s} best val_accuracy = {max(h.history['val_accuracy']):.4f}")

## ✍️ Practice
1. Compare convergence speed (epochs to 90% val accuracy) of the BatchNorm+Nadam model vs a plain SGD model.
2. Explain why "dying ReLU" happens, and why Leaky ReLU fixes it.
3. Does the Dropout model's validation accuracy improve vs the plain model? Does it need more epochs?
4. Debug: fine-tuning ALL layers of a pretrained model on 500 images with the pretraining learning rate hurts accuracy — what's wrong, what 2 things would you try?

In [ ]:
# Your code / notes here